In [1]:
import tkinter as tk
import random
import matplotlib.pyplot as plt
from collections import Counter



# CAR (CORE)

class Car:
    def __init__(self, car_id, car_type, arrival_time):
        self.id = car_id
        self.type = car_type
        self.arrival_time = arrival_time
        self.park_time = None


# PAYMENT


def process_payment(amount):
    methods = ["Card", "Wallet", "Cash"]
    return f"{random.choice(methods)} Payment: {amount:.2f}"


# PRICING

def calculate_price(car_type, duration, occupancy):
    base = 5

    if car_type == "VIP":
        base *= 2
    elif car_type == "Electric":
        base *= 0.5

    if occupancy > 0.8:
        base *= 1.2

    if duration > 10:
        base += (duration - 10) * 2

    return base * duration



# PARKING SYSTEM

class ParkingSystem:
    def __init__(self):
        self.capacity = 7
        self.floor1 = []
        self.floor2 = []
        self.queue = []

        self.time = 0
        self.car_counter = 0

        self.revenue = 0
        self.last_cost = 0
        self.last_payment = ""

        self.occupancy_history = []
        self.queue_history = []
        self.car_type_history = []

        # Accuracy
        self.correct_predictions = 0
        self.total_predictions = 0

    def generate_car(self):
        r = random.random()
        if r < 0.2:
            t = "VIP"
        elif r < 0.4:
            t = "Electric"
        else:
            t = "Normal"

        self.car_counter += 1
        return Car(f"C{self.car_counter}", t, self.time)

    def predict_peak(self):
        return len(self.queue) > 3

    def actual_peak(self):
        occ = (len(self.floor1)+len(self.floor2))/(self.capacity*2)
        return occ > 0.7

    def update_accuracy(self):
        pred = self.predict_peak()
        actual = self.actual_peak()

        self.total_predictions += 1
        if pred == actual:
            self.correct_predictions += 1

    def get_accuracy(self):
        if self.total_predictions == 0:
            return 0
        return self.correct_predictions / self.total_predictions

    def park_car(self, car):
        if len(self.floor1) < self.capacity:
            self.floor1.append(car)
        elif len(self.floor2) < self.capacity:
            self.floor2.append(car)
        else:
            self.queue.append(car)
            return False

        car.park_time = self.time
        return True

    def remove_car(self):
        if self.floor2 and random.random() < 0.5:
            car = self.floor2.pop(0)
        elif self.floor1:
            car = self.floor1.pop(0)
        else:
            return

        duration = self.time - car.park_time
        occ = (len(self.floor1)+len(self.floor2))/(self.capacity*2)

        cost = calculate_price(car.type, duration, occ)
        payment = process_payment(cost)

        self.last_cost = cost
        self.last_payment = payment
        self.revenue += cost

        self.car_type_history.append(car.type)

    def move_queue(self):
        while self.queue and (len(self.floor1) < self.capacity or len(self.floor2) < self.capacity):
            self.park_car(self.queue.pop(0))

    def step(self):
        self.time += 1

        if random.random() < 0.7:
            self.park_car(self.generate_car())

        if random.random() < 0.5:
            self.remove_car()

        self.move_queue()

        occ = (len(self.floor1)+len(self.floor2))/(self.capacity*2)
        self.occupancy_history.append(occ)
        self.queue_history.append(len(self.queue))

        self.update_accuracy()



# UI

class App:
    def __init__(self, root):
        self.system = ParkingSystem()
        self.running = False

        root.title("🚗 Smart Parking Pro MAX")
        root.geometry("900x700")
        root.configure(bg="#0f172a")

        tk.Label(root, text="Smart Parking Dashboard",
                 font=("Arial", 20, "bold"),
                 bg="#0f172a", fg="#38bdf8").pack(pady=10)

        frame = tk.Frame(root, bg="#0f172a")
        frame.pack()

        tk.Button(frame, text="▶ Start", command=self.start, bg="#22c55e", width=15).grid(row=0, column=0)
        tk.Button(frame, text="⏸ Stop", command=self.stop, bg="#ef4444", width=15).grid(row=0, column=1)
        tk.Button(frame, text="📊 Graphs", command=self.graphs, bg="#3b82f6", width=15).grid(row=0, column=2)
        tk.Button(frame, text="📑 Report", command=self.report, bg="#a855f7", width=15).grid(row=0, column=3)

        self.label = tk.Label(root, text="", bg="#0f172a", fg="white", font=("Arial", 14))
        self.label.pack(pady=10)

        self.cost_label = tk.Label(root, text="Last Cost: 0",
                                  bg="#0f172a", fg="#facc15",
                                  font=("Arial", 14, "bold"))
        self.cost_label.pack()

        self.payment_label = tk.Label(root, text="Last Payment: -",
                                     bg="#0f172a", fg="#38bdf8")
        self.payment_label.pack()

        self.acc_label = tk.Label(root, text="Accuracy: 0%",
                                  bg="#0f172a", fg="#22c55e",
                                  font=("Arial", 14, "bold"))
        self.acc_label.pack()

        self.log = tk.Text(root, height=15, bg="#020617", fg="#38bdf8")
        self.log.pack(fill="both", padx=20, pady=10)

    def start(self):
        self.running = True
        self.loop()

    def stop(self):
        self.running = False

    def loop(self):
        if not self.running:
            return

        self.system.step()

        cars = len(self.system.floor1)+len(self.system.floor2)
        queue = len(self.system.queue)
        occ = self.system.occupancy_history[-1]

        acc = self.system.get_accuracy() * 100

        text = f"Cars: {cars} | Queue: {queue} | Revenue: {self.system.revenue:.2f}"
        self.label.config(text=text)

        self.cost_label.config(text=f"Last Cost: {self.system.last_cost:.2f}")
        self.payment_label.config(text=f"Last Payment: {self.system.last_payment}")
        self.acc_label.config(text=f"Accuracy: {acc:.2f}%")

        if queue > 5 or occ > 0.8:
            self.label.config(fg="red")
        else:
            self.label.config(fg="white")

        self.log.insert(tk.END, text + "\n")
        self.log.see(tk.END)

        root.after(500, self.loop)

    
    # GRAPHS 
    
    def graphs(self):
        plt.figure()

        plt.subplot(2,1,1)
        plt.plot(self.system.occupancy_history)
        plt.title("Occupancy Over Time")

        plt.subplot(2,1,2)
        plt.plot(self.system.queue_history)
        plt.title("Queue Over Time")

        plt.tight_layout()
        plt.show()

    def report(self):
        if not self.system.car_type_history:
            return

        count = Counter(self.system.car_type_history)

        plt.figure()
        plt.pie(count.values(), labels=count.keys(), autopct='%1.1f%%')
        plt.title("Car Types Distribution")
        plt.show()


# RUN
root = tk.Tk()
app = App(root)
root.mainloop()